In [ ]:
import scanpy as sc
import scanpy.external as sce
import pandas as pd
import numpy as np
import os
import shutil
import triku as tk
import matplotlib.pyplot as plt
import matplotlib as mpl
import subprocess
from scipy.sparse import csr_matrix
from IPython.display import display, HTML
import mygene as mg
import seaborn as sns
from tqdm import tqdm
import phate
# from tqdm.notebook import tqdm

from bokeh.io import show, output_notebook, reset_output

from scipy.sparse import csr_matrix, csc_matrix

reset_output()
output_notebook()

In [ ]:
from pyfuncs.general import preprocessing_adata_sub, plot_volcano


In [ ]:
from cellassign import assign_cats

In [ ]:
magma = [plt.get_cmap('magma')(i) for i in np.linspace(0,1, 80)]
magma[0] = (0.88, 0.88, 0.88, 1)
magma = mpl.colors.LinearSegmentedColormap.from_list("", magma[:65])

seed = 0

In [ ]:
from matplotlib import font_manager, rcParams

sns.set_style("white")

font_path = "src/fonts/texgyreheros-regular.otf" # Alternative
font_path = "/usr/share/texmf/fonts/opentype/public/tex-gyre/texgyreheros-regular.otf"

mpl.rcParams.update({'font.size': 22})


# Path to the Helvetica font file
custom_font = font_manager.FontProperties(fname=font_path)

# Add the font to Matplotlib's font manager
font_manager.fontManager.addfont(font_path)

# Set the font globally
rcParams['font.family'] = custom_font.get_name()
rcParams['font.size'] = 16
rcParams['figure.dpi'] = 300

# Oprescu adata load

In [ ]:
adata_oprescu_FAP = sc.read_h5ad('data/processed/oprescu_processed_clean_FAP.h5ad')

In [ ]:
adata_oprescu_FAP_NI  = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'Noninjured']
adata_oprescu_FAP_D05 = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'X0.5.DPI']
adata_oprescu_FAP_D2  = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'X2.DPI']
adata_oprescu_FAP_D35 = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'X3.5.DPI']
adata_oprescu_FAP_D5  = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'X5.DPI']
adata_oprescu_FAP_D10 = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'X10.DPI']
adata_oprescu_FAP_D21 = adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == 'X21.DPI']

# Assigning labels to all-state adata

In [ ]:
list_sub_names = adata_oprescu_FAP.obs['batch'].cat.categories
list_sub_adatas = [adata_oprescu_FAP_NI, adata_oprescu_FAP_D05, adata_oprescu_FAP_D2, adata_oprescu_FAP_D35, adata_oprescu_FAP_D5, adata_oprescu_FAP_D10, adata_oprescu_FAP_D21]

adata_oprescu_FAP.obs['sub_population'] = ''

for name, adata in zip(list_sub_names, list_sub_adatas):
    adata_oprescu_FAP.obs.loc[adata.obs_names, 'sub_population'] = adata.obs['sub_population'].astype(str)

In [ ]:
sc.pl.umap(adata_oprescu_FAP, color='sub_population')

fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for idx, batch in enumerate(adata_oprescu_FAP.obs['batch'].unique()):
    sc.pl.umap(adata_oprescu_FAP, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5)
    sc.pl.umap(adata_oprescu_FAP[adata_oprescu_FAP.obs['batch'] == batch], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8)
    axs.flatten()[idx].set_title(batch)

**What do we see?**

After running individual analyses on each day, we see that the unassigned cells cluster together, which is a good sign that FAP assignment was successful. 

# What's next?

Now that we've got the FAP and unassigned clusters, we want to answer two questions.
* What functions do the unassigned clusters of each day have?
* What relationships within each day and between days exist between the clusters.

For the functions, we are going to asume that cells in different areas of the shared UMAP will have different DEGs. If a cluster is shared across several days, we're first going to extract the DEGs of the combined cells altogether, since it's going to reveal a more general tendency, rather than a day-specific tendency (we can get these DEGs later on if we want).

**First, we are going to clean the tenocyte and myonuclei clusters, because we want to focus the analysis on FAPs. Also, At *Uninjured* state there are few tenocytes to perform the day-to-day analysis confidently.** 

To ensure that all myonuclei and tenocytes are removed, we are going to remove specific clusters via leiden.

In [ ]:
sc.pl.umap(adata_oprescu_FAP, color='major_population')

In [ ]:
sc.tl.leiden(adata_oprescu_FAP, resolution=1.3)
sc.pl.umap(adata_oprescu_FAP, color='leiden')

### Remove Myonuclei and Tenocyte clusters 

In [ ]:
MYO_CLUSTERS = ['16']
TENO_CLUSTERS = ['5']

adata_oprescu_onlyFAP = adata_oprescu_FAP[~(
    adata_oprescu_FAP.obs['leiden'].isin(MYO_CLUSTERS + TENO_CLUSTERS) | 
    adata_oprescu_FAP.obs['major_population'].isin(['Myonuclei', 'Tenocyte']) |
    adata_oprescu_FAP.obs['sub_population'].isin(['Myonuclei', 'Teno_B', 'Teno_C', 'Teno_D'])
)
]

In [ ]:
adata_oprescu_onlyFAP

In [ ]:
sc.pp.filter_genes(adata_oprescu_onlyFAP, min_counts=1)

sc.pp.pca(adata_oprescu_onlyFAP, random_state=seed, n_comps=50)
sce.pp.harmony_integrate(adata_oprescu_onlyFAP, 'batch', max_iter_harmony=20, random_state=seed, )
sc.pp.neighbors(adata_oprescu_onlyFAP, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_onlyFAP) ** 0.5 // 3), metric='cosine')
tk.tl.triku(adata_oprescu_onlyFAP)
sc.pp.pca(adata_oprescu_onlyFAP, random_state=seed, n_comps=50)
sce.pp.harmony_integrate(adata_oprescu_onlyFAP, 'batch', max_iter_harmony=20, random_state=seed)
sc.pp.neighbors(adata_oprescu_onlyFAP, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_onlyFAP) ** 0.5 // 3), metric='cosine')

sc.tl.umap(adata_oprescu_onlyFAP, min_dist=0.15, random_state=seed)
sc.pl.umap(adata_oprescu_onlyFAP, color='sub_population')

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(30, 6))
for idx, batch in enumerate(adata_oprescu_onlyFAP.obs['sub_population'].unique()):
    sc.pl.umap(adata_oprescu_onlyFAP, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5)
    sc.pl.umap(adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['sub_population'] == batch], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8)
    axs.flatten()[idx].set_title(batch)



In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for idx, batch in enumerate(adata_oprescu_onlyFAP.obs['batch'].unique()):
    sc.pl.umap(adata_oprescu_onlyFAP, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5)
    sc.pl.umap(adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['batch'] == batch], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8)
    axs.flatten()[idx].set_title(batch)

axs.flatten()[-1].set_axis_off()

In [ ]:
sc.tl.leiden(adata_oprescu_onlyFAP, resolution=1.2, key_added='leiden_FAPs')
sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_FAPs')

In [ ]:
adata_oprescu_onlyFAP_U = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['sub_population'] == 'unassigned']


sc.pp.filter_genes(adata_oprescu_onlyFAP_U, min_counts=1)

sc.pp.pca(adata_oprescu_onlyFAP_U, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_oprescu_onlyFAP_U, 'batch', max_iter_harmony=30, random_state=seed, )
sc.pp.neighbors(adata_oprescu_onlyFAP_U, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_onlyFAP_U) ** 0.5 // 2), metric='cosine')
tk.tl.triku(adata_oprescu_onlyFAP_U)
sc.pp.pca(adata_oprescu_onlyFAP_U, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_oprescu_onlyFAP_U, 'batch', max_iter_harmony=30, random_state=seed)
sc.pp.neighbors(adata_oprescu_onlyFAP_U, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_onlyFAP_U) ** 0.5 // 2), metric='cosine')

In [ ]:
sc.tl.umap(adata_oprescu_onlyFAP_U, min_dist=0.2, random_state=seed)
sc.tl.leiden(adata_oprescu_onlyFAP_U, resolution=0.3)

sc.pl.umap(adata_oprescu_onlyFAP_U, color='batch')
sc.pl.umap(adata_oprescu_onlyFAP_U, color='leiden')


In [ ]:
adata_oprescu_onlyFAP.obs['leiden_U'] = 'rest'
adata_oprescu_onlyFAP.obs.loc[adata_oprescu_onlyFAP_U.obs_names, 'leiden_U'] = adata_oprescu_onlyFAP_U.obs.loc[adata_oprescu_onlyFAP_U.obs_names, 'leiden']
sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_U')

## Can we separate sub clusters of the first days?

We see that the majority of cells ad day 0.5 and 2 are located within clusters 1, 6 and 12. We are going to subset these clusters and also 2 and 9, to account for days 3.5 and 5, and analyze their structure.

In [ ]:
SELECTED_CLUSTERS = ['1', '6', '12',]
adata_D05_2 = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['leiden_FAPs'].isin(SELECTED_CLUSTERS)]


sc.pp.filter_genes(adata_D05_2, min_counts=1)

sc.pp.pca(adata_D05_2, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_D05_2, 'batch', max_iter_harmony=30, random_state=seed, )
sc.pp.neighbors(adata_D05_2, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_D05_2) ** 0.5 // 2), metric='cosine')
tk.tl.triku(adata_D05_2)
sc.pp.pca(adata_D05_2, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_D05_2, 'batch', max_iter_harmony=30, random_state=seed)
sc.pp.neighbors(adata_D05_2, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_D05_2) ** 0.5 // 2), metric='cosine')

In [ ]:
sc.tl.umap(adata_D05_2, min_dist=0.3, random_state=seed)
sc.pl.umap(adata_D05_2, color='sub_population')
sc.pl.umap(adata_D05_2, color='batch')

In [ ]:
sc.tl.leiden(adata_D05_2, resolution=0.3)
sc.pl.umap(adata_D05_2, color='leiden')

In [ ]:
adata_oprescu_onlyFAP.obs['leiden_D05_2'] = 'rest'
adata_oprescu_onlyFAP.obs.loc[adata_D05_2.obs_names, 'leiden_D05_2'] = adata_D05_2.obs.loc[adata_D05_2.obs_names, 'leiden']
sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_D05_2')

We can also incorporate clusters 2 and 9, which begin to be represented for days D3.5 and D5

In [ ]:
CLUSTERS_D35_D5 = ['2', '9', '8', '10'] #8 and 10 just to make sure

adata_oprescu_onlyFAP.obs['leiden_D05_5'] = 'rest'
adata_oprescu_onlyFAP.obs.loc[adata_D05_2.obs_names, 'leiden_D05_5'] = adata_D05_2.obs.loc[adata_D05_2.obs_names, 'leiden']

idx = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['leiden_FAPs'].isin(CLUSTERS_D35_D5)].obs_names
adata_oprescu_onlyFAP.obs.loc[idx, 'leiden_D05_5'] = '4'

sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_D05_5')
sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_U')

Based on the joint nomenclature from leiden_D05_5 and leiden_U, we are going to perform a joint population naming based on othe following set of rules, into a new obs.

In [ ]:
adata_oprescu_onlyFAP.obs['leiden_S'] = 'Non-S'

S0_index = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['leiden_U'] =='0'].obs_names
adata_oprescu_onlyFAP.obs.loc[S0_index, 'leiden_S'] = 'S0'

S1A_index = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['leiden_U'] =='1'].obs_names
adata_oprescu_onlyFAP.obs.loc[S1A_index, 'leiden_S'] = 'S1'

S1B_index = adata_oprescu_onlyFAP[(adata_oprescu_onlyFAP.obs['leiden_U'] =='rest') &
                                  (adata_oprescu_onlyFAP.obs['leiden_D05_5'] =='0') ].obs_names
adata_oprescu_onlyFAP.obs.loc[S1B_index, 'leiden_S'] = 'S1'

S2A_index = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['leiden_U'] =='2'].obs_names
adata_oprescu_onlyFAP.obs.loc[S2A_index, 'leiden_S'] = 'S2'

S2B_index = adata_oprescu_onlyFAP[(adata_oprescu_onlyFAP.obs['leiden_U'] =='rest') & 
                                  (adata_oprescu_onlyFAP.obs['leiden_D05_5'] =='4')].obs_names
adata_oprescu_onlyFAP.obs.loc[S2B_index, 'leiden_S'] = 'S2'

SX_index = adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['leiden_D05_5'] =='3'].obs_names
adata_oprescu_onlyFAP.obs.loc[SX_index, 'leiden_S'] = 'SX'

In [ ]:
sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_S')
sc.pl.umap(adata_oprescu_onlyFAP, color='sub_population')

In [ ]:
display(adata_oprescu_onlyFAP[(adata_oprescu_onlyFAP.obs['batch'] == 'X10.DPI') & \
    (adata_oprescu_onlyFAP.obs['leiden_S'] == 'Non-S') &
    (adata_oprescu_onlyFAP.obs['sub_population'] == 'unassigned')])

sc.pl.umap(adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['batch'] == 'X10.DPI'], color='leiden_S')
sc.pl.umap(adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['batch'] == 'X10.DPI'], color='sub_population')

### Running PHATE on FAP data

In [ ]:
phate_operator = phate.PHATE(knn=5, decay=None, t='auto', gamma=0.5, n_jobs=-2, verbose=True)
adata_oprescu_onlyFAP.obsm['X_phate'] = phate_operator.fit_transform(adata_oprescu_onlyFAP.X)

In [ ]:
sce.pl.phate(adata_oprescu_onlyFAP, color=['batch'], ncols=2)

fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for idx, batch in enumerate(adata_oprescu_onlyFAP.obs['batch'].unique()):
    sce.pl.phate(adata_oprescu_onlyFAP, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5)
    sce.pl.phate(adata_oprescu_onlyFAP[adata_oprescu_onlyFAP.obs['batch'] == batch], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8)
    axs.flatten()[idx].set_title(batch)

axs.flatten()[-1].set_axis_off()

We see an almost clear separation of data across days, with D3 and D5 being together. One option Thus would be to separate the days into that option.

Also, from the UMAP we see that data from days 0.5 and 2 clusters in 4 main populations.

In [ ]:
sce.pl.phate(adata_oprescu_onlyFAP, color=['leiden_S'], ncols=2)


In [ ]:
adata_oprescu_onlyFAP.obs['leiden_S']

## Presence of clusters in individual-day adatas

In [ ]:
THRESHOLD_PER = 1.5
THRESHOLD_CELLS = 40

def get_merged_clusters(adata_day, FAP_over_S=False):
    print(adata_day.obs['batch'].values[0])

    intersect_idx = np.intersect1d(adata_day.obs_names, adata_oprescu_onlyFAP.obs_names)
    adata_day.obs.loc[:, 'leiden_S'] = 'Non-S'
    adata_day.obs.loc[intersect_idx, 'leiden_S'] = adata_oprescu_onlyFAP.obs.loc[intersect_idx, 'leiden_S'].astype(str)

    # Option A: FAP populations over S - the criteria is that the original nomenclature of FAPs is correct
    if FAP_over_S:
        adata_day.obs['merged_clusters'] = adata_day.obs['sub_population'].astype(str).copy()
        unassigned_cells = adata_day[adata_day.obs['sub_population'] == 'unassigned'].obs_names
        adata_day.obs.loc[unassigned_cells, 'merged_clusters'] = adata_day.obs.loc[unassigned_cells, 'leiden_S'].astype(str)


    # Option B: S over FAP populations - the criteria is that the S naming is more important than FAP population names
    else:
        adata_day.obs['merged_clusters'] = adata_day.obs['leiden_S'].astype(str).copy()
        unassigned_cells = adata_day[adata_day.obs['merged_clusters'] == 'Non-S'].obs_names
        adata_day.obs.loc[unassigned_cells, 'merged_clusters'] = adata_day.obs.loc[unassigned_cells, 'sub_population'].astype(str)
        


    adata_day = adata_day[~((adata_day.obs['leiden_S'] == 'Non-S') &
    (adata_day.obs['sub_population'] == 'unassigned'))
    ]

    value_counts = pd.DataFrame(adata_day.obs['merged_clusters'].value_counts())
    value_counts['percentage'] = 100 * value_counts / value_counts.sum()
    display(value_counts)


    selected_types = value_counts[(value_counts['percentage'] > THRESHOLD_PER) & 
                      (value_counts['count'] > THRESHOLD_CELLS)].index
    display(selected_types)
    adata_day = adata_day[adata_day.obs['merged_clusters'] != 'unassigned']
    adata_day = adata_day[adata_day.obs['merged_clusters'].isin(selected_types)]


    preprocessing_adata_sub(adata_day, integrate=False, n_comps=30)

    sc.tl.umap(adata_day)

    sc.pl.umap(adata_day, color='merged_clusters')

    return adata_day

In [ ]:
# We are going to remove minor populations to remove complexity in graph/trajectory 
# analyses, and unassigned cells, becaue they are minority and if they appear in 
# >D5 they are irrelevant.


adata_oprescu_FAP_NI = get_merged_clusters(adata_oprescu_FAP_NI, FAP_over_S=True)
adata_oprescu_FAP_D05 = get_merged_clusters(adata_oprescu_FAP_D05)
adata_oprescu_FAP_D2 = get_merged_clusters(adata_oprescu_FAP_D2)
adata_oprescu_FAP_D35 = get_merged_clusters(adata_oprescu_FAP_D35)
adata_oprescu_FAP_D5 = get_merged_clusters(adata_oprescu_FAP_D5)
adata_oprescu_FAP_D10 = get_merged_clusters(adata_oprescu_FAP_D10)
adata_oprescu_FAP_D21 = get_merged_clusters(adata_oprescu_FAP_D21, FAP_over_S=True)

In [ ]:
adata_oprescu_FAP = sc.concat([adata_oprescu_FAP_NI, adata_oprescu_FAP_D05, adata_oprescu_FAP_D2, 
                    adata_oprescu_FAP_D35, adata_oprescu_FAP_D5, adata_oprescu_FAP_D10, adata_oprescu_FAP_D21])

In [ ]:
intersect_cells = np.intersect1d(adata_oprescu_FAP.obs_names, adata_oprescu_onlyFAP.obs_names)

adata_oprescu_onlyFAP = adata_oprescu_onlyFAP[intersect_cells, :]

adata_oprescu_onlyFAP.obs['merged_clusters'] = ''
adata_oprescu_onlyFAP.obs.loc[intersect_cells, 'merged_clusters'] = adata_oprescu_FAP.obs.loc[intersect_cells, 'merged_clusters']

## DEG characterization:

We are going to apply two strategies for DEGs:
- Day-based DEGs: we are going to get DEGs based on each day.
- Cluster-based DEGs: we are going to set clusters according to the subclustering we performed earlier, of 0-4 + rest

### Day-based DEGs

In [ ]:
dict_DEGs_day = {}

sc.tl.rank_genes_groups(adata_oprescu_onlyFAP, groupby='batch', use_raw=False)

for batch in adata_oprescu_onlyFAP.obs['batch'].cat.categories:
    display(batch)
    df_pvals = plot_volcano(adata_oprescu_onlyFAP, batch, pval_threshold=1e-30, lfc_threshold=0.25, topn=30, bottomn=0, return_df=True, xlim=(0, 4))
    
    dict_DEGs_day[batch] = df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:50]
    
    sce.pl.phate(adata_oprescu_onlyFAP, color=df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:10], cmap=magma, use_raw=False, ncols=5)
    
    fig, axs = plt.subplots(2, 5, figsize=(20, 4))
    for idx, gene in enumerate(df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:10]):
        sc.pl.violin(adata_oprescu_onlyFAP, keys=gene, 
                    groupby='batch', use_raw=False, ax=axs.flatten()[idx], show=False)
    plt.tight_layout()
    plt.show()

    df_pvals.to_csv(f"results/oprescu/DEGS_DAY_{batch.replace('.', '-')}.csv", index=None)

### Cluster-based DEGs

In [ ]:
sc.pl.umap(adata_oprescu_onlyFAP, color='leiden_S')

In [ ]:
df_pvals

In [ ]:
dict_DEGs_S = {}

sc.tl.rank_genes_groups(adata_oprescu_onlyFAP, groupby='leiden_S', use_raw=False, method='t-test_overestim_var')

for S_cluster in adata_oprescu_onlyFAP.obs['leiden_S'].cat.categories:
    display(S_cluster)
    df_pvals = plot_volcano(adata_oprescu_onlyFAP, S_cluster, pval_threshold=1e-15, lfc_threshold=0.25, topn=30, bottomn=0, return_df=True, xlim=(0, 4))

    dict_DEGs_S[S_cluster] = df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:50]

    sc.pl.umap(adata_oprescu_onlyFAP, color=df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:20], cmap=magma, use_raw=False, ncols=5)

    fig, axs = plt.subplots(4, 5, figsize=(20, 10))
    for idx, gene in enumerate(df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:20]):
        sc.pl.violin(adata_oprescu_onlyFAP, keys=gene, 
                    groupby='leiden_S', use_raw=False, ax=axs.flatten()[idx], show=False)
    plt.tight_layout()
    plt.show()

    df_pvals.to_csv(f'results/oprescu/DEGS_S_{S_cluster}.csv', index=None)

In [ ]:
def plot_jaccard(dict_rows, dict_cols):
    df_jaccard = pd.DataFrame(index = dict_rows.keys(), columns = dict_cols.keys(), dtype=np.float32)

    for row, genes_row in dict_rows.items():
        for col, genes_col in dict_cols.items():
            if row != col:
                jac = 100 * len(np.intersect1d(genes_row, genes_col)) / len(np.union1d(genes_row, genes_col))
                df_jaccard.loc[row, col] = jac
            else:
                df_jaccard.loc[row, col] = np.nan

    return df_jaccard

In [ ]:
sns.heatmap(plot_jaccard(dict_DEGs_day, dict_DEGs_S), annot=True, cmap='Blues')

In [ ]:
sns.heatmap(plot_jaccard(dict_DEGs_S, dict_DEGs_S), annot=True, cmap='Blues')

In [ ]:
sns.heatmap(plot_jaccard(dict_DEGs_day, dict_DEGs_day), annot=True, cmap='Blues')

### Analysing the relationships between cell types of different days

We are going to create a merged category with already detected populations and unassigned clusters. Also, if specific cells with no assignment are missing, we are going to remove them from the adata to reduce the noise.

In [ ]:
sc.pl.umap(adata_oprescu_onlyFAP, color=['leiden_S', 'sub_population'])

In [ ]:
# Generate the cleaned adata without unassigned cells

adata_oprescu_onlyFAP_clean = adata_oprescu_onlyFAP[~((adata_oprescu_onlyFAP.obs['sub_population'] == 'unassigned') & 
                                                    (adata_oprescu_onlyFAP.obs['leiden_S'] == 'Non-S'))]

In [ ]:
adata_oprescu_onlyFAP

In [ ]:
adata_oprescu_onlyFAP_clean

#### Intra-day relationships

In [ ]:
THRESHOLD = 50  # We set a high threshold to ensure that the populations are "consistent" and are the most important ones

In [ ]:
dict_adata_clean_days = {}

for datapoint in adata_oprescu_onlyFAP_clean.obs['batch'].cat.categories:
    print('DATAPOINT', datapoint)
    adata_oprescu_onlyFAP_clean_datapoint = adata_oprescu_onlyFAP_clean[adata_oprescu_onlyFAP_clean.obs['batch'] == datapoint]

    value_counts = adata_oprescu_onlyFAP_clean_datapoint.obs['merged_clusters'].value_counts()
    display(value_counts) 
    selected_clusters = value_counts[value_counts > THRESHOLD].index.values
    adata_oprescu_onlyFAP_clean_datapoint = adata_oprescu_onlyFAP_clean_datapoint[adata_oprescu_onlyFAP_clean_datapoint.obs['merged_clusters'].isin(selected_clusters)]

    preprocessing_adata_sub(adata_oprescu_onlyFAP_clean_datapoint, integrate=False)
    sc.tl.umap(adata_oprescu_onlyFAP_clean_datapoint, min_dist=0.1, random_state=seed)
    sc.pl.umap(adata_oprescu_onlyFAP_clean_datapoint, color='merged_clusters')

    dict_adata_clean_days[datapoint] = adata_oprescu_onlyFAP_clean_datapoint.copy()

In [ ]:
for datapoint in dict_adata_clean_days.keys():
    adata_datapoint = dict_adata_clean_days[datapoint]
    print(datapoint)
    sc.pl.umap(adata_datapoint, color='merged_clusters')
    try:
        sc.tl.paga(adata_datapoint, groups='merged_clusters')
        sc.pl.paga(adata_datapoint)
    except:
        print(f"{datapoint} failed")

    

#### Inter-day relationships

In [ ]:
list_datapoint_0 = adata_oprescu_onlyFAP_clean.obs['batch'].cat.categories
list_datapoint_1 = list_datapoint_0[1:].to_list() + [list_datapoint_0[0]]

for idx in range(len(list_datapoint_0)):
    datapoint_0 = list_datapoint_0[idx]
    datapoint_1 = list_datapoint_1[idx]

    print('\nPAIR ', datapoint_0, '|', datapoint_1)

    adata_pair = sc.concat([dict_adata_clean_days[datapoint_0], dict_adata_clean_days[datapoint_1]])
    
    preprocessing_adata_sub(adata_pair)
    sc.tl.umap(adata_pair)
    sc.pl.umap(adata_pair, color='merged_clusters')
    sc.pl.umap(adata_pair, color='batch')

    adata_pair.obs['batch-merged_clusters'] = adata_pair.obs['batch'].astype(str) + '-' + adata_pair.obs['merged_clusters'].astype(str)

    sc.tl.paga(adata_pair, groups='batch-merged_clusters')

    

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import scanpy as sc

SEP = "||"
END_SUFFIX = "__end"


def make_node_id(row_key, cluster):
    return f"{row_key}{SEP}{cluster}"


def split_node_id(node_id):
    return node_id.split(SEP, 1)


def sort_key(x):
    x = str(x)
    try:
        return (0, int(x))
    except ValueError:
        try:
            return (1, float(x))
        except ValueError:
            return (2, x)
        

def build_global_graph_layout_from_dict(
    dict_adata_clean_days,
    batch_order=None,
    cluster_col="merged_clusters",
    cluster_order=None,
    append_missing_clusters=True,
    x_spacing=2.2,
    y_spacing=2.2,
    duplicate_first_at_end=True,
):
    """
    Layout global:
    - cada batch en una fila Y
    - cada cluster ocupa SIEMPRE la misma X en todas las filas
    - puedes fijar un orden manual con cluster_order
    - opcionalmente duplica el primer batch al final
    """

    if batch_order is None:
        batch_order = sorted(dict_adata_clean_days.keys(), key=sort_key)
    else:
        batch_order = [str(x) for x in batch_order]

    first_batch = batch_order[0]

    # Sacar clusters globales reales
    global_clusters = set()
    for batch in batch_order:
        global_clusters.update(
            dict_adata_clean_days[batch].obs[cluster_col].astype(str).unique().tolist()
        )
    global_clusters = set(map(str, global_clusters))

    # Resolver orden de clusters
    if cluster_order is None:
        cluster_order = sorted(global_clusters, key=sort_key)
    else:
        cluster_order = [str(x) for x in cluster_order]

        # comprobar si hay clusters en cluster_order que no existen
        unknown_clusters = [cl for cl in cluster_order if cl not in global_clusters]
        if len(unknown_clusters) > 0:
            raise ValueError(
                f"Estos clusters de cluster_order no existen en los datos: {unknown_clusters}"
            )

        # añadir al final los que faltan, si se quiere
        if append_missing_clusters:
            missing_clusters = sorted(
                [cl for cl in global_clusters if cl not in cluster_order],
                key=sort_key
            )
            cluster_order = cluster_order + missing_clusters
        else:
            # usar SOLO los que hayas indicado
            cluster_order = [cl for cl in cluster_order if cl in global_clusters]

    # Posición X fija para cada cluster
    xs = np.arange(len(cluster_order), dtype=float) * x_spacing
    if len(xs) > 0:
        xs = xs - xs.mean()
    cluster_x = dict(zip(cluster_order, xs))

    # Filas visuales
    row_specs = [(b, b, b) for b in batch_order]
    if duplicate_first_at_end:
        row_specs.append((f"{first_batch}{END_SUFFIX}", first_batch, first_batch))

    G = nx.Graph()
    pos = {}
    row_y = {}

    for i, (row_key, actual_batch, display_label) in enumerate(row_specs):
        ad = dict_adata_clean_days[actual_batch]

        present_clusters = (
            ad.obs[cluster_col]
            .astype(str)
            .drop_duplicates()
            .tolist()
        )

        # respetar el orden global del eje X
        present_clusters = [cl for cl in cluster_order if cl in present_clusters]

        y = -i * y_spacing
        row_y[row_key] = y

        for cluster in present_clusters:
            x = cluster_x[cluster]
            node = make_node_id(row_key, cluster)
            G.add_node(
                node,
                row_key=row_key,
                batch=actual_batch,
                display_batch=display_label,
                cluster=cluster,
            )
            pos[node] = (x, y)

    return G, pos, row_specs, row_y, first_batch, cluster_order, cluster_x

In [ ]:
def add_edges_from_paga_pair(
    G,
    adata_pair,
    day0,
    day1,
    first_batch,
    group_col="batch_merged_clusters",
    threshold=0.0,
    combine="max",
):
    categories = adata_pair.obs[group_col].cat.categories.astype(str).tolist()

    conn = adata_pair.uns["paga"]["connectivities"]
    df_adj = pd.DataFrame(conn.toarray(), index=categories, columns=categories)

    nodes_day0 = [x for x in categories if split_node_id(x)[0] == str(day0)]
    nodes_day1 = [x for x in categories if split_node_id(x)[0] == str(day1)]

    cross = df_adj.loc[nodes_day0, nodes_day1]

    # Decidir qué fila visual usar
    src_row_key = str(day0)
    dst_row_key = str(day1)

    # Si el destino es el primer batch y venimos del último día,
    # usamos la copia visual de abajo
    if str(day1) == str(first_batch) and str(day0) != str(first_batch):
        dst_row_key = f"{first_batch}{END_SUFFIX}"

    for src_actual in cross.index:
        src_cluster = split_node_id(src_actual)[1]
        src_plot = make_node_id(src_row_key, src_cluster)

        for dst_actual, w in cross.loc[src_actual].items():
            w = float(w)
            if w <= threshold:
                continue

            dst_cluster = split_node_id(dst_actual)[1]
            dst_plot = make_node_id(dst_row_key, dst_cluster)

            if G.has_edge(src_plot, dst_plot):
                if combine == "max":
                    G[src_plot][dst_plot]["weight"] = max(G[src_plot][dst_plot]["weight"], w)
                elif combine == "sum":
                    G[src_plot][dst_plot]["weight"] += w
            else:
                G.add_edge(
                    src_plot,
                    dst_plot,
                    weight=w,
                    pair=f"{day0}->{day1}",
                )

    return cross

In [ ]:
def plot_paga_graph(
    G,
    pos,
    row_specs,
    row_y,
    cluster_order=None,
    cluster_x=None,
    figsize=(18, 10),
    node_size=1000,
    edge_scale=2.0,
    label_size=10,
    edge_mode="grayscale_by_weight",   # "uniform_gray" o "grayscale_by_weight"
    edge_gray=0.75,                    # usado si edge_mode="uniform_gray"
    show_cluster_labels_top=True,
):
    fig, ax = plt.subplots(figsize=figsize)

    # -----------------
    # Dibujar edges
    # -----------------
    weights = [d["weight"] for _, _, d in G.edges(data=True)]
    if len(weights) == 0:
        w_min, w_max = 0.0, 1.0
    else:
        w_min, w_max = min(weights), max(weights)

    def normalize_weight(w):
        if w_max == w_min:
            return 1.0
        return (w - w_min) / (w_max - w_min)

    for u, v, d in G.edges(data=True):
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        w = d["weight"]
        w_norm = normalize_weight(w)

        linewidth = 0.4 + edge_scale * w

        if edge_mode == "uniform_gray":
            color = str(edge_gray)  # 0 negro, 1 blanco
            alpha = 0.9
        else:
            # finos -> gris clarito; gruesos -> gris oscuro
            gray = 0.82 - 0.67 * w_norm
            gray = min(max(gray, 0.12), 0.82)
            color = str(gray)
            alpha = 0.95

        ax.plot(
            [x1, x2],
            [y1, y2],
            linewidth=linewidth,
            color=color,
            alpha=alpha,
            zorder=1,
        )

    # -----------------
    # Colores de nodos por batch real
    # -----------------
    actual_batches = []
    for _, actual_batch, _ in row_specs:
        if actual_batch not in actual_batches:
            actual_batches.append(actual_batch)

    cmap = plt.get_cmap("tab20")
    color_map = {batch: cmap(i % 20) for i, batch in enumerate(actual_batches)}

    # -----------------
    # Dibujar nodos
    # -----------------
    for row_key, actual_batch, display_label in row_specs:
        nodes = [
            n for n, data in G.nodes(data=True)
            if data["row_key"] == row_key
        ]
        xs = [pos[n][0] for n in nodes]
        ys = [pos[n][1] for n in nodes]

        ax.scatter(
            xs,
            ys,
            s=node_size,
            color=color_map[actual_batch],
            edgecolor="black",
            linewidth=0.8,
            zorder=2,
        )

        for n in nodes:
            x, y = pos[n]
            ax.text(
                x,
                y,
                G.nodes[n]["cluster"],
                ha="center",
                va="center",
                fontsize=label_size,
                zorder=3,
            )

    # -----------------
    # Etiquetas eje Y
    # -----------------
    ax.set_yticks([row_y[row_key] for row_key, _, _ in row_specs])
    ax.set_yticklabels([display_label for _, _, display_label in row_specs])

    # -----------------
    # Etiquetas de clusters arriba
    # -----------------
    if show_cluster_labels_top and (cluster_order is not None) and (cluster_x is not None):
        top_y = max(y for _, y in pos.values()) + 1.2
        for cl in cluster_order:
            ax.text(
                cluster_x[cl],
                top_y,
                cl,
                ha="center",
                va="bottom",
                fontsize=9,
                rotation=45,
            )

    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_ylabel("batch")
    ax.set_title("PAGA inter-day cluster graph")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
merge_column = 'merged_clusters'

In [ ]:
list_datapoint_0 = adata_oprescu_onlyFAP_clean.obs["batch"].cat.categories.astype(str).tolist()
list_datapoint_1 = list_datapoint_0[1:] + [list_datapoint_0[0]]

G, pos, row_specs, row_y, first_batch, cluster_order, cluster_x = build_global_graph_layout_from_dict(
    dict_adata_clean_days,
    batch_order=list_datapoint_0,
    cluster_order=[ 'SX', 'S0', 'S1',  'S2', 'Krano', 'FAP_A', 'FAP_B', 'FAP_CDEF', ],
    cluster_col=merge_column,
    duplicate_first_at_end=True,
)

cross_mats = {}

for day0, day1 in zip(list_datapoint_0[:], list_datapoint_1[:]):
    print("\nPAIR", day0, "|", day1)

    adata_pair = sc.concat(
        [dict_adata_clean_days[day0], dict_adata_clean_days[day1]],
        join="outer"
    )

    preprocessing_adata_sub(adata_pair)

    adata_pair.obs[f"batch_{merge_column}"] = (
        adata_pair.obs["batch"].astype(str)
        + SEP
        + adata_pair.obs[merge_column].astype(str)
    ).astype("category")

    sc.tl.paga(adata_pair, groups=f"batch_{merge_column}")

    cross = add_edges_from_paga_pair(
        G,
        adata_pair,
        day0=day0,
        day1=day1,
        first_batch=first_batch,
        group_col=f"batch_{merge_column}",
        threshold=0.95,
        combine="max",
    )

    cross_mats[(day0, day1)] = cross

plot_paga_graph(
    G,
    pos,
    row_specs,
    row_y,
    cluster_order=cluster_order,
    cluster_x=cluster_x,
    edge_mode="grayscale_by_weight",   # o "uniform_gray"
    edge_gray=0.75,
)

In [ ]:
list_datapoint_0 = adata_oprescu_onlyFAP_clean.obs["batch"].cat.categories.astype(str).tolist()
list_datapoint_1 = list_datapoint_0[1:] + [list_datapoint_0[0]]

G, pos, row_specs, row_y, first_batch, cluster_order, cluster_x = build_global_graph_layout_from_dict(
    dict_adata_clean_days,
    batch_order=list_datapoint_0,
    cluster_order=[ 'SX', 'S0', 'S1',  'S2', 'Krano', 'FAP_A', 'FAP_B', 'FAP_CDEF', ],
    cluster_col=merge_column,
    duplicate_first_at_end=True,
)

cross_mats = {}

for day0, day1 in zip(list_datapoint_0[:], list_datapoint_1[:]):
    print("\nPAIR", day0, "|", day1)

    adata_pair = sc.concat(
        [dict_adata_clean_days[day0], dict_adata_clean_days[day1]],
        join="outer"
    )

    preprocessing_adata_sub(adata_pair)

    adata_pair.obs[f"batch_{merge_column}"] = (
        adata_pair.obs["batch"].astype(str)
        + SEP
        + adata_pair.obs[merge_column].astype(str)
    ).astype("category")

    sc.tl.paga(adata_pair, groups=f"batch_{merge_column}")

    cross = add_edges_from_paga_pair(
        G,
        adata_pair,
        day0=day0,
        day1=day1,
        first_batch=first_batch,
        group_col=f"batch_{merge_column}",
        threshold=0.45,
        combine="max",
    )

    cross_mats[(day0, day1)] = cross

plot_paga_graph(
    G,
    pos,
    row_specs,
    row_y,
    cluster_order=cluster_order,
    cluster_x=cluster_x,
    edge_mode="grayscale_by_weight",   # o "uniform_gray"
    edge_gray=0.75,
)

In [ ]:
dict_adata_clean_days.items()

In [ ]:
# ── Parámetros ─────────────────────────────────────────────────────────

pop_key       = "merged_clusters"
time_key      = "batch"
top_n_per_pair = 7      # top N aristas por par de días
min_threshold  = 0.05   # filtro mínimo anti-ruido
x_spacing      = 2.5
y_spacing      = 2.5
cluster_order  = ["SX", "S0", "S1", "S2", "Krano", "FAP_A", "FAP_B", "FAP_CDEF"]

day_order = ["Noninjured", "X0.5.DPI", "X2.DPI", "X3.5.DPI", "X5.DPI", "X10.DPI", "X21.DPI"]

# Filas visuales: el primero se duplica al final
row_keys = [(d, d) for d in day_order] + [(day_order[0] + "__end", day_order[0])]

# Pares contiguos + cierre del ciclo
pairs = list(zip(day_order[:-1], day_order[1:]))
pairs += [(day_order[-1], day_order[0])]

# ── Posiciones ─────────────────────────────────────────────────────────

existing = set(cl for ad in dict_adata_clean_days.values()
               for cl in ad.obs[pop_key].astype(str).unique())
cluster_order = [c for c in cluster_order if c in existing]
cluster_order += sorted([c for c in existing if c not in cluster_order])

cluster_x = {cl: i * x_spacing for i, cl in enumerate(cluster_order)}
row_y     = {row_key: -i * y_spacing for i, (row_key, _) in enumerate(row_keys)}

# ── Contar células por (día, población) ────────────────────────────────

count_dict = {}
for day, ad in dict_adata_clean_days.items():
    for pop, n in ad.obs[pop_key].astype(str).value_counts().items():
        count_dict[(day, pop)] = n

# ── Construir grafo y nodos ─────────────────────────────────────────────

G   = nx.Graph()
pos = {}

for row_key, actual_day in row_keys:
    ad = dict_adata_clean_days[actual_day]
    for pop in ad.obs[pop_key].astype(str).unique():
        node = (row_key, pop)
        G.add_node(node, row_key=row_key, day=actual_day, pop=pop)
        pos[node] = (cluster_x[pop], row_y[row_key])

# ── Calcular PAGA por par y añadir top N aristas ───────────────────────

for day0, day1 in pairs:
    print(f"PAGA: {day0} → {day1}")

    adata_pair = sc.concat(
        [dict_adata_clean_days[day0], dict_adata_clean_days[day1]],
        join="outer"
    )
    preprocessing_adata_sub(adata_pair)

    adata_pair.obs["batch_pop"] = (
        adata_pair.obs[time_key].astype(str) + "||" +
        adata_pair.obs[pop_key].astype(str)
    ).astype("category")

    sc.tl.paga(adata_pair, groups="batch_pop")

    cats = adata_pair.obs["batch_pop"].cat.categories.astype(str).tolist()
    conn = pd.DataFrame(
        adata_pair.uns["paga"]["connectivities"].toarray(),
        index=cats, columns=cats
    )

    src_nodes = [c for c in cats if c.startswith(str(day0) + "||")]
    tgt_nodes = [c for c in cats if c.startswith(str(day1) + "||")]

    src_row = day0
    tgt_row = day1 if (day1 != day_order[0] or day0 == day_order[0]) else day_order[0] + "__end"

    # Recoger todas las aristas del par por encima del mínimo
    edges_this_pair = []
    for src in src_nodes:
        pop_src = src.split("||")[1]
        for tgt in tgt_nodes:
            w = float(conn.loc[src, tgt])
            if w <= min_threshold:
                continue
            pop_tgt = tgt.split("||")[1]
            n_src = (src_row, pop_src)
            n_tgt = (tgt_row, pop_tgt)
            if n_src in G and n_tgt in G:
                edges_this_pair.append((n_src, n_tgt, w))

    # Ordenar y quedarse con el top N
    edges_this_pair.sort(key=lambda x: x[2], reverse=True)
    for n_src, n_tgt, w in edges_this_pair[:top_n_per_pair]:
        if G.has_edge(n_src, n_tgt):
            G[n_src][n_tgt]["weight"] = max(G[n_src][n_tgt]["weight"], w)
        else:
            G.add_edge(n_src, n_tgt, weight=w)


In [ ]:

fig, ax = plt.subplots(figsize=(16, 10))

cmap      = plt.get_cmap("tab10")
day_color = {d: cmap(i % 10) for i, d in enumerate(day_order)}

weights  = [d["weight"] for _, _, d in G.edges(data=True)]
w_min, w_max = (min(weights), max(weights)) if weights else (0, 1)

for u, v, d in G.edges(data=True):
    w_norm = (d["weight"] - w_min) / (w_max - w_min + 1e-9)
    gray   = 1 - 0.70 * w_norm
    lw     = 0.5 + 4.0 * w_norm
    x1, y1 = pos[u]
    x2, y2 = pos[v]
    ax.plot([x1, x2], [y1, y2], lw=lw, color=str(gray), alpha=0.9, zorder=1)

max_count = max(count_dict.values())
min_size, max_size = 100, 2000

for node in G.nodes():
    row_key, pop = node
    actual_day   = G.nodes[node]["day"]
    x, y  = pos[node]
    n     = count_dict.get((actual_day, pop), 1)
    size  = min_size + (max_size - min_size) * (n / max_count)
    ax.scatter(x, y, s=size, color=day_color[actual_day],
               edgecolors="black", linewidths=0.8, zorder=2)
    # ax.text(x, y, pop, ha="center", va="center", fontsize=7, zorder=3)

# Eje Y: etiquetas de las filas visuales
ytick_positions = [row_y[rk] for rk, _ in row_keys]
ytick_labels    = [actual_day for _, actual_day in row_keys]
ax.set_yticks(ytick_positions)
ax.set_yticklabels(ytick_labels)

bottom_y = min(y for _, y in pos.values()) - 1.2
for cl in cluster_order:
    ax.text(cluster_x[cl], bottom_y, cl,
            ha="center", va="top", fontsize=10, rotation=0)

ax.set_xticks([])
ax.set_title("PAGA inter-day cluster graph", fontsize=13)
ax.set_ylabel("batch")
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig("paga_transition_graph.pdf", bbox_inches="tight")
plt.show()